In [ ]:
# https://www.kaggle.com/datasets/wyattowalsh/basketball/data
# https://github.com/mpope9/nba-sql/blob/master/image/NBA-ER.jpg

In [4]:
from urllib.request import urlopen
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
import warnings
import re
import psycopg2

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=FutureWarning)

In [5]:
options = webdriver.FirefoxOptions()
options.add_argument('-headless')
driver = webdriver.Firefox(options = options)

The geckodriver version (0.33.0) detected in PATH at geckodriver.EXE might not be compatible with the detected firefox version (130.0.1.967); currently, geckodriver 0.35.0 is recommended for firefox 130.*, so it is advised to delete the driver in PATH and retry


In [ ]:
try:
    # Establece la conexión con la base de datos
    connection = psycopg2.connect(
        dbname="mydatabase",
        user="myuser",
        password="mysecretpassword",
        host="localhost",  # normalmente es 'localhost' si es local
        port="5432"  # normalmente es 5432
    )
    
    # Crea un cursor para realizar operaciones en la base de datos
    cursor = connection.cursor()
except Exception as error:
    print(f"Error al conectar con la base de datos: {error}")

In [3]:
def getTableIDS(url):
    driver.get(url)
    tables_id = driver.find_elements(By.XPATH, "//table[@id]")
    list_id_tables = []
    for table in tables_id:
        table_id = table.get_attribute("id")
        list_id_tables.append(table_id)
    return list_id_tables

In [ ]:
def delete_unnamed_columns(df):
    df = df.loc[:, ~df.columns.str.contains('Unnamed')]
    return df

In [ ]:
def get_keys_dictionary(diccionario):
    keys = set(diccionario.keys())
    for values in diccionario.values():
        if isinstance(values, dict):
            keys.update(get_keys_dictionary(values))
    return keys

In [ ]:
def check_missing_values(dictionary):
    for key, value in dictionary.items():
        if isinstance(value, dict):
            print(f"Recorriendo diccionario bajo la clave '{key}':")
            check_missing_values(value)  
        elif isinstance(value, pd.DataFrame): 
            print(f"Revisando DataFrame bajo la clave '{key}':")
            
            if value.isnull().values.any():
                print("¡Hay valores nulos en el DataFrame!")
                print(value)
            
            unnamed_columns = [col for col in value.columns if 'Unnamed' in col]
            if unnamed_columns:
                print(f"¡El DataFrame tiene columnas 'Unnamed': {unnamed_columns}")

            empty_columns = [col for col in value.columns if value[col].empty]
            if empty_columns:
                print(f"¡El DataFrame tiene columnas vacías: {empty_columns}")

In [ ]:
# NBA Standings que es como quedó la season con todos los equipos
years = list(range(2022, 2023))
dictionary_of_teams = {}
# Itera a través de cada identificador de tabla y guarda en un DataFrame
dataframes = []
for year in years:
    dictionary_of_teams[year] = {}
    url = f'https://www.basketball-reference.com/leagues/NBA_{year}_standings.html'
    table_ids = getTableIDS(url)
    time.sleep(3)
    for table_id in table_ids:
        dictionary_of_teams[year][table_id] = {}
        table_element = driver.find_element(By.ID, table_id)
        table_html = table_element.get_attribute('outerHTML')
        df = pd.read_html(table_html, header=0)[0]
        if table_id == 'expanded_standings':
            new_header = df.iloc[0]
            df = df[1:]
            df.columns = new_header
            df.reset_index(drop=True, inplace=True)
            dictionary_of_teams[year][table_id] = df
        else:    
            dictionary_of_teams[year][table_id] = df

In [ ]:
# De aqui para arriba tenemos las estadisticas generales de los equipos
# De aqui para abajo sacaremos las estadisticas de los jugadores por equipo

In [4]:
years = list(range(2022, 2023))
dictionary_of_players = {}
dictionary_of_players_playoffs = {}
teams_NBA_list = ['ATL']
# teams_NBA_list =  ['ATL', 'BOS', 'BRK', 'CHO', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND','LAC','LAL','MEM','MIA','MIL','MIN','NOP', 'NYK','OKC', 'ORL','PHI', 
#                    'PHO', 'POR','SAC','SAS','TOR','UTA','WAS']

for team in teams_NBA_list:
    dictionary_of_players[team] = {}
    dictionary_of_players_playoffs[team] = {}
    for year in years:
        dictionary_of_players[team][year] = {}
        dictionary_of_players_playoffs[team][year] = {}
        url = f'https://www.basketball-reference.com/teams/{team}/{year}.html'
        table_ids = getTableIDS(url)
        time.sleep(3)
        for table_id in table_ids:
            if 'playoffs' in table_id:
                dictionary_of_players_playoffs[team][year][table_id] = {}
                table_element = driver.find_element(By.ID, table_id)
                table_html = table_element.get_attribute('outerHTML')
                df = pd.read_html(table_html, header=0)[0]    
                if  table_id == 'playoffs_pbp':
                    new_header = df.iloc[0]
                    df = df[1:]
                    df.columns = new_header
                    df.reset_index(drop=True, inplace=True)
                else:    
                    dictionary_of_players_playoffs[team][year][table_id] = df
            else:
                dictionary_of_players[team][year][table_id] = {}
                table_element = driver.find_element(By.ID, table_id)
                table_html = table_element.get_attribute('outerHTML')
                df = pd.read_html(table_html, header=0)[0]    
                if table_id == 'adj_shooting' or table_id == 'shooting' or table_id == 'pbp':
                    new_header = df.iloc[0]
                    df = df[1:]
                    df.columns = new_header
                    df.reset_index(drop=True, inplace=True)
                else:    
                    dictionary_of_players[team][year][table_id] = df
        

In [3]:
class PlayerScraper:
        def __init__(self):
            self.url = 'https://www.basketball-reference.com/teams/',
            self.teams_NBA_list = 'ATL',
            self.years = '2024',
    
        def get_players_team_year(self):
            dictionary_of_teams = {}
            for nba_team in self.teams_NBA_list:
                dictionary_of_teams[nba_team] = {} 
                for year in self.years: 
                    dictionary_of_teams[nba_team][year] = [] 
                    url = f'https://www.basketball-reference.com/teams/{nba_team}/{year}.html' 
                    response = requests.get(url) 
                    soup = BeautifulSoup(response.content, 'html.parser') 
                    table = soup.find('table', {'id': 'advanced'}) 
                    if table: 
                        headers = [th.text.strip() for th in table.find('thead').find_all('th')] 
                        rows = [ 
                            {headers[i]: cell.text.strip() for i, cell in enumerate(tr.find_all(['th', 'td']))} 
                            for tr in table.find('tbody').find_all('tr') 
                        ] 
                        dictionary_of_teams[nba_team][year] = rows 
            return dictionary_of_teams

In [4]:
scraper_player = PlayerScraper()
players_data_by_team = scraper_player.get_players_team_year()

In [5]:
for team, year in dictionary_of_players.items():
    for year, tables in year.items():
        print(tables.keys())


# DE AQUI VER QUE TABLA INTERESA, E INTENTAR VER COMO ESCTRUCTURAR LA BASE DE DATOS
# PERO ANTES DE NADA CREAR TABLAS CON LOS DATOS QUE QUERAMOS

dict_keys(['roster', 'team_and_opponent', 'team_misc', 'per_game', 'totals', 'per_minute', 'per_poss', 'advanced', 'adj_shooting', 'shooting', 'pbp', 'salaries2'])


In [16]:
class PlayerScraper:
        def __init__(self):
            self.url = 'https://www.basketball-reference.com/teams/',
            self.teams_NBA_list = 'ATL',
            self.years = '2024',
        
        def get_team_regular_season_results(self):
            dictionary_of_teams = {}
            for nba_team in self.teams_NBA_list:
                dictionary_of_teams[nba_team] = {}
                for year in self.years:
                    dictionary_of_teams[nba_team][year] = []
                    url = f'https://www.basketball-reference.com/teams/{nba_team}/{year}_games.html'
                    response = requests.get(url)
                    soup = BeautifulSoup(response.content, 'html.parser')
                    table = soup.find('table', {'id': 'games'})
                    if table:
                        headers = [th['data-stat'] for th in table.find('thead').find_all('th')]
                        for tr in table.find('tbody').find_all('tr'):
                            # Ignorar los tr que tienen la clase 'thead'
                            if 'thead' in tr.get('class', []):
                                continue
                            # Ignorar filas que tienen un th con el valor del primer header
                            if tr.find('th') and tr.find('th').text.strip() == headers[0]:
                                continue
                            row_data = {headers[i]: cell.text.strip() for i, cell in enumerate(tr.find_all(['th', 'td']))}
                            dictionary_of_teams[nba_team][year].append(row_data)
            return dictionary_of_teams


In [17]:
scraper_player = PlayerScraper()
team_advanced = scraper_player.get_team_regular_season_results()

In [7]:
for team, year in team_advanced.items():
    print(year)

{'2024': [{'g': '1', 'date_game': 'Wed, Oct 25, 2023', 'game_start_time': '7:00p', 'network': '', 'box_score_text': 'Box Score', 'game_location': '@', 'opp_name': 'Charlotte Hornets', 'game_result': 'L', 'overtimes': '', 'pts': '110', 'opp_pts': '116', 'wins': '0', 'losses': '1', 'game_streak': 'L 1', 'attendance': '16,129', 'game_duration': '2:22', 'game_remarks': ''}, {'g': '2', 'date_game': 'Fri, Oct 27, 2023', 'game_start_time': '7:30p', 'network': '', 'box_score_text': 'Box Score', 'game_location': '', 'opp_name': 'New York Knicks', 'game_result': 'L', 'overtimes': '', 'pts': '120', 'opp_pts': '126', 'wins': '0', 'losses': '2', 'game_streak': 'L 2', 'attendance': '17,692', 'game_duration': '2:11', 'game_remarks': ''}, {'g': '3', 'date_game': 'Sun, Oct 29, 2023', 'game_start_time': '7:00p', 'network': '', 'box_score_text': 'Box Score', 'game_location': '@', 'opp_name': 'Milwaukee Bucks', 'game_result': 'W', 'overtimes': '', 'pts': '127', 'opp_pts': '110', 'wins': '1', 'losses': '2'

In [18]:
team_advanced['ATL']['2024'][20]

{'g': '21',
 'date_game': 'Fri, Dec 8, 2023',
 'game_start_time': '7:00p',
 'network': '',
 'box_score_text': 'Box Score',
 'game_location': '@',
 'opp_name': 'Philadelphia 76ers',
 'game_result': 'L',
 'overtimes': '',
 'pts': '114',
 'opp_pts': '125',
 'wins': '9',
 'losses': '12',
 'game_streak': 'L 3',
 'attendance': '19,746',
 'game_duration': '2:13',
 'game_remarks': ''}

In [8]:
for nba_team, games in player_advanced.items():
    for year, games in games.items():
        for stats in games:
            print(stats.keys())

In [ ]:
#celda de limpieza de datos
dictionary_of_players['ATL'][2022]['roster'] = dictionary_of_players['ATL'][2022]['roster'].drop(columns=['Unnamed: 6'])
dictionary_of_players['ATL'][2022]['team_and_opponent'] = dictionary_of_players['ATL'][2022]['team_and_opponent'].rename(columns={'Unnamed: 0': ''})
dictionary_of_players['ATL'][2022]['team_misc'].columns = dictionary_of_players['ATL'][2022]['team_misc'].iloc[0]
dictionary_of_players['ATL'][2022]['team_misc'] = dictionary_of_players['ATL'][2022]['team_misc'][1:]
dictionary_of_players['ATL'][2022]['per_poss'] = dictionary_of_players['ATL'][2022]['per_poss'] .drop(columns=['Unnamed: 27'])
dictionary_of_players['ATL'][2022]['advanced'] = dictionary_of_players['ATL'][2022]['advanced'].drop(columns=['Unnamed: 17', 'Unnamed: 22'])